[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-13-review-architecture.ipynb#scrollTo=ll000001)

---
# Day 13 · Architecture Review & Best Practices
**certified-journeys / hamilton-certified** · Day 13 · Review

> **Goal for today:** Review the complete Hamilton design vocabulary, recognize common anti-patterns, and apply the module-splitting decision guide to a real-world feature pipeline.

In [ ]:
%pip install -q sf-hamilton

## The Hamilton Design Vocabulary — A 14-Day Recap

| Concept | Days | One-line summary |
|---|---|---|
| Function = node | 1 | Every `def` in a Hamilton module is a DAG node |
| Parameters = edges | 1 | Dependencies declared by parameter name, not explicit wiring |
| Driver | 2 | Executes the graph; single entry point for config, modules, adapters |
| `@config.when` | 3 | Conditional implementation; activates one branch per config value |
| `@tag` | 3 | Metadata label; queryable at build or run time |
| `@extract_columns` | 3 | DataFrame → column Series; one-to-many expand |
| Feature pipeline | 4–5 | Layered: raw → derived → model_inputs; one module per layer |
| Module namespacing | 5 | `subdag` lets you reuse a module under a different namespace |
| `@does` | 6 | DRY: delegate implementation to a shared function |
| `@check_output` | 6 | Runtime data contract on node output |
| Testing: 3 levels | 7 | Unit (no Driver), config branch, integration with overrides |
| `@pipe` / `@mutate` | 8 | Sequential chaining and non-invasive post-processing |
| AsyncDriver | 9 | Concurrent I/O-bound nodes |
| `Parallelizable/Collect` | 9 | Fan-out / fan-in on many independent inputs |
| Polars backend | 10 | Swap `pd.Series` → `pl.Series`; use `@config.when(backend=...)` |
| Lifecycle hooks | 11 | Cross-cutting observability without touching function code |
| Materializers | 12 | Execute + persist in one call; custom `DataSaver` |


## Anti-Patterns — Recognizing Bad Hamilton Code

Even experienced Hamilton users make these mistakes.

In [ ]:
import sys, types
import numpy as np
import pandas as pd
from hamilton import driver
from hamilton.function_modifiers import tag, extract_columns, config

# ── Anti-pattern 1: God function ──────────────────────────────────────────────
# BAD: one function does everything — untestable, can't reuse partial outputs
def all_features_bad(raw_data: pd.DataFrame) -> pd.DataFrame:
    """❌ BAD: one node for all features — no intermediate reuse."""
    age = raw_data['age']
    age_z = (age - age.mean()) / age.std()
    spend = raw_data['spend']
    spend_log = np.log1p(spend)
    tenure = raw_data['tenure']
    tenure_yr = tenure / 12.0
    return pd.DataFrame({'age_z': age_z, 'spend_log': spend_log, 'tenure_yr': tenure_yr})

# GOOD: each transformation is its own node
def age_zscore(age: pd.Series) -> pd.Series:    # ✓ testable in isolation
    return (age - age.mean()) / age.std()

def spend_log(spend: pd.Series) -> pd.Series:   # ✓ reusable downstream
    return np.log1p(spend)

def tenure_years(tenure: pd.Series) -> pd.Series:
    return tenure / 12.0

print('Anti-pattern 1: God function')
print('  BAD  → all_features_bad():  1 node, 3 outputs hidden inside a DataFrame')
print('  GOOD → age_zscore, spend_log, tenure_years: 3 nodes, each testable')

In [ ]:
# ── Anti-pattern 2: Side effects inside nodes ─────────────────────────────────
import time

def age_zscore_bad(age: pd.Series) -> pd.Series:
    """❌ BAD: side effect inside a transform node."""
    result = (age - age.mean()) / age.std()
    result.to_csv('/tmp/debug_age.csv')  # ← BAD: side effect
    return result

def age_zscore_good(age: pd.Series) -> pd.Series:
    """✓ GOOD: pure transform, no side effects."""
    return (age - age.mean()) / age.std()

# Why bad? Side effects in nodes:
#   - Make tests slow (I/O)
#   - Create hidden dependencies (filesystem state)
#   - Break parallel execution (concurrent writes)
#   - Use materialisers instead for intentional I/O

print('Anti-pattern 2: Side effects in nodes')
print('  BAD  → to_csv inside a transform (implicit I/O)')
print('  GOOD → dr.materialize(to.csv(...)) for intentional persistence')

# ── Anti-pattern 3: Inputs that are too specific ─────────────────────────────
# BAD: function accepts full DataFrame when it only needs one column
def age_zscore_too_wide(raw_data: pd.DataFrame) -> pd.Series:
    """❌ BAD: accepts raw_data but only uses 'age'."""
    return (raw_data['age'] - raw_data['age'].mean()) / raw_data['age'].std()

# GOOD: narrow inputs = clearer dependencies = better testability
def age_zscore_narrow(age: pd.Series) -> pd.Series:
    """✓ GOOD: narrow input — just the column it needs."""
    return (age - age.mean()) / age.std()

print('\nAnti-pattern 3: Too-wide inputs')
print('  BAD  → accepts raw_data (whole DataFrame) when only one column is needed')
print('  GOOD → accept age: pd.Series — forces explicit extraction upstream')

## Module Splitting — The Decision Guide

When should you create a new module? Apply these rules in order:

| Rule | Split? |
|---|---|
| Different lifecycle (ingest runs once per day, features run per request) | **Yes** |
| Different owners (data engineering vs ML team) | **Yes** |
| Used by multiple pipelines (shared cleaning logic) | **Yes** |
| > 20 functions in one module | **Probably** |
| Just feels big / long | **No** |
| Functions only used together, never separately | **No** |

**Standard 4-module layout:**
```
ingest.py      → raw_data, raw_labels
clean.py       → *_cleaned  (data contracts)
features.py    → *_zscore, *_log, *_encoded  (ML features)
model_inputs.py → feature_matrix, label_vector
```

In [ ]:
# Demonstrate the 4-module layout with a minimal end-to-end pipeline

# ingest.py
def raw_data(source_path: str) -> pd.DataFrame:
    """Load raw data. In production: accepts a db connection."""
    rng = np.random.default_rng(int(hash(source_path)) % 2**31)
    N = 100
    return pd.DataFrame({
        'age': rng.integers(18, 75, N).astype(float),
        'spend': np.where(rng.random(N) < 0.05, -1.0, rng.exponential(80, N)),
        'churned': rng.integers(0, 2, N),
    })

# clean.py
@extract_columns('age', 'spend', 'churned')
def raw_df_extracted(raw_data: pd.DataFrame) -> pd.DataFrame:
    return raw_data

def age_clean(age: pd.Series) -> pd.Series:
    return age.clip(18, 100).fillna(age.median())

def spend_clean(spend: pd.Series) -> pd.Series:
    return spend.clip(lower=0)

# features.py
@tag(feature_type='numerical')
def age_norm(age_clean: pd.Series) -> pd.Series:
    return (age_clean - age_clean.mean()) / age_clean.std()

@tag(feature_type='numerical')
def spend_norm(spend_clean: pd.Series) -> pd.Series:
    return (spend_clean - spend_clean.mean()) / spend_clean.std()

@tag(feature_type='boolean')
def is_high_spender(spend_clean: pd.Series) -> pd.Series:
    return (spend_clean > spend_clean.quantile(0.75)).astype(float)

# model_inputs.py
def feature_matrix(age_norm: pd.Series, spend_norm: pd.Series, is_high_spender: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({'age_norm': age_norm, 'spend_norm': spend_norm, 'is_high_spender': is_high_spender})

def label_vector(churned: pd.Series) -> pd.Series:
    return churned.astype(float)

# Wire all 4 modules
full_module = types.ModuleType('full_pipeline')
for fn in [raw_data, raw_df_extracted, age_clean, spend_clean,
           age_norm, spend_norm, is_high_spender, feature_matrix, label_vector]:
    setattr(full_module, fn.__name__, fn)
sys.modules['full_pipeline'] = full_module

dr = driver.Builder().with_modules(full_module).build()
result = dr.execute(
    ['feature_matrix', 'label_vector'],
    inputs={'source_path': 'customers.parquet'}
)

X = result['feature_matrix']
y = result['label_vector']
print(f'Feature matrix: {X.shape}')
print(f'Label vector:   {y.shape}, churn rate: {y.mean():.1%}')
print(f'Features: {list(X.columns)}')
print('✓ Full 4-module pipeline executed')

## Decorator Cheat Sheet — Final Reference

In [ ]:
cheat_sheet = [
    # (decorator, purpose, example)
    ('@tag(key=val)',          'Metadata labels on a node',      "@tag(feature_type='numerical')"),
    ('@extract_columns(cols)', 'DataFrame → N named Series',     "@extract_columns('age','spend')"),
    ('@config.when(k=v)',      'Conditional implementation',      "@config.when(env='prod')"),
    ('@does(fn)',              'DRY — delegate to shared fn',    "@does(_zscore)"),
    ('@check_output(...)',     'Runtime data contract',          "@check_output(allow_nans=False)"),
    ('@pipe(step(fn),...)',    'Sequential transforms on one obj',"@pipe(step(fill), step(clip))"),
    ('@mutate(node=fn)',       'Post-process another node',      "@mutate(score_raw=_cap)"),
    ('@resolve(when=...)',     'Config-driven decorator choice', "@resolve(when=CONFIG_EAGER,...)"),
    ('Parallelizable[T]',     'Fan-out to N branches',          "def ids() -> Parallelizable[int]"),
    ('Collect[T]',            'Fan-in from N branches',         "def all(s: Collect[float])"),
]

print(f'{"Decorator":<26}  {"Purpose":<38}  Example')
print('─' * 100)
for dec, purpose, example in cheat_sheet:
    print(f'{dec:<26}  {purpose:<38}  {example}')

---
## Day 13 review summary

| Anti-pattern | Fix |
|---|---|
| God function — everything in one node | Split to one function per concept |
| Side effects inside nodes (print, to_csv) | Use materializers for intentional I/O |
| Too-wide inputs (full DataFrame) | Accept narrow types (single Series) |
| No `@check_output` on data boundaries | Add contracts at ingest→clean, clean→features |
| All modules in one file | Split by lifecycle, owner, or reuse |  

> **Tip:** The sign of a well-structured Hamilton pipeline: every function is testable in isolation with a single Series input, no Driver required.

---
## What's next
**Day 14** → Capstone: build an end-to-end ML feature pipeline from scratch, applying everything from Days 1–13.

Mark Day 13 complete in your [tracker](../index.html).